In [179]:
import re
import pandas as pd 

from pathlib import Path


pd.set_option("display.max_colwidth", None)

RAW_FILE = Path("../data/raw/pao_metadata/Book1.xlsx")
REVIEW_FILE = Path("../data/interim/pao_label_review.csv")
OUTPUT_FILE = Path("../data/processed/pao_metadata_v1.parquet")

print(f"raw file", RAW_FILE.exists(),  "\nreview file: ",  REVIEW_FILE.exists(), "\nOutput File: ", OUTPUT_FILE.exists())

raw file True 
review file:  True 
Output File:  False


In [180]:
HEADER_ROWS = {
    "COMMUNICATION": 2,
    "CLEARANCE": 0,
    "AOsMEMOs": 2,
    "LEAVES": 0,
}

sheets={}


for name, header in HEADER_ROWS.items():
    df = pd.read_excel(RAW_FILE, sheet_name=name, header=header)
    df.columns = df.columns.astype(str).str.strip()
    sheets[name] = df
    print(f"{name:15} rows={df.shape[0]:5} columns={df.shape[1]}")


COMMUNICATION   rows= 2285 columns=12
CLEARANCE       rows=  702 columns=16
AOsMEMOs        rows=   14 columns=6
LEAVES          rows=  657 columns=9


In [181]:
for name, df in sheets.items():
    print(f"\n---{name}---")
    print(df.columns.tolist())

for name, df in sheets.items():
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]

    if not unnamed:
        print(f"{name:15} no Unnamed Columns")
        continue

    print(f"\n{name}:")
    for col in unnamed:
        filled=df[col].notna().sum()
        print(f" {col:12} values: {filled}")
        if filled > 0:
            display(df.loc[df[col].notna(), [col]].head(5))


comm = sheets["COMMUNICATION"]
rows = comm["Unnamed: 10"].notna()


display(comm.loc[rows, ["SUBJECT","DESCRIPTION", "Unnamed: 10"]])

subject = comm["SUBJECT"].fillna("").astype(str).str.strip().str.upper()
desc    = comm["DESCRIPTION"].fillna("").astype(str).str.strip().str.upper()

same       = subject == desc
desc_empty = desc == ""

# subject_in_desc = [s in d for s, d in zip(subject, desc)]
subject_in_desc = [s != "" and s in d for s, d in zip(subject, desc)]

print("rows:                        ", len(comm))
print("DESCRIPTION same as SUBJECT: ", same.sum())
print("DESCRIPTION empty:           ", desc_empty.sum())
print("SUBJECT contained in DESC:   ", sum(subject_in_desc))


---COMMUNICATION---
['DATE', 'CONTROL#', 'SUBJECT', 'DESCRIPTION', 'TYPE', 'OFFICE/PERSON FROM', 'OFFICE/UNIT CONCERNED', 'REMARKS', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

---CLEARANCE---
['DATE', 'CONTROL#', 'FULL N AME', 'OFFICE', 'PURPOSE', 'REMARKS', 'RELEASING DATE', 'C/O', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15']

---AOsMEMOs---
['DATE RECEIVED', 'DOCUMENT DATE', 'AO/MEMO/EO #', 'SUBJECT', 'OFFICE/UNIT/PERSON FROM', 'REMARKS']

---LEAVES---
['DATE FILED', 'TRANSMITAL DATE', 'CONTROL#', 'NAME', 'TYPE OF LEAVE', 'REMARKS', 'DAY/S APPLIED', 'DATE RECEIVED', 'STATUS']

COMMUNICATION:
 Unnamed: 9   values: 0
 Unnamed: 10  values: 3


,Unnamed: 10
446,"THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,Orientation on Konsulta Package: An Initiative Towards a Functional Health Care Provider Network (HCPN)


 Unnamed: 11  values: 3


,Unnamed: 11
446,"THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN)



CLEARANCE:
 Unnamed: 9   values: 0
 Unnamed: 10  values: 0
 Unnamed: 11  values: 0
 Unnamed: 12  values: 0
 Unnamed: 13  values: 0
 Unnamed: 14  values: 0
 Unnamed: 15  values: 1


,Unnamed: 15
272,


AOsMEMOs        no Unnamed Columns
LEAVES          no Unnamed Columns


,SUBJECT,DESCRIPTION,Unnamed: 10
446,"THE AMOUNT THREE HUNDRED FORTY NINE MILLION SIX HUNDRED FIFTY THREE THOUSAND TWO HUNDRED SEVENTEEN PESOS ONLY (349,653,217.00) HAS BEEN CREDITED TO YOUR ACCOUNT ON JANUARY 31, 2025.","THE AMOUNT THREE HUNDRED FORTY NINE MILLION SIX HUNDRED FIFTY THREE THOUSAND TWO HUNDRED SEVENTEEN PESOS ONLY (349,653,217.00) HAS BEEN CREDITED TO YOUR ACCOUNT ON JANUARY 31, 2025.","THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"INVITATION - 2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY.","INVITATION - 2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY.","2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN),ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN),Orientation on Konsulta Package: An Initiative Towards a Functional Health Care Provider Network (HCPN)


rows:                         2285
DESCRIPTION same as SUBJECT:  1604
DESCRIPTION empty:            0
SUBJECT contained in DESC:    1722


In [182]:
#remove empty columns and rows
for name, df in sheets.items():
    before = df.shape

    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    sheets[name]=df
    print(f"{name:15} before={before} after={df.shape}")


COMMUNICATION   before=(2285, 12) after=(2285, 9)
CLEARANCE       before=(702, 16) after=(702, 9)
AOsMEMOs        before=(14, 6) after=(14, 6)
LEAVES          before=(657, 9) after=(653, 9)


In [183]:
#give every row a permanent ID
for name, df in sheets.items():
    df=df.reset_index(drop=True)
    df["row_id"] = name + "-" + df.index.astype(str)
    sheets[name]= df

display(sheets["COMMUNICATION"][["row_id", "SUBJECT"]].head(3))


#adds IDs like COMMUNICATION-0 that stay sttached to the same record so saved review decisions always match


,row_id,SUBJECT
0,COMMUNICATION-0,REQUEST TO RENDER OVERTIME
1,COMMUNICATION-1,PFC MEETING
2,COMMUNICATION-2,NOTICE OF ASSIGNMENT/ GRANT OF SECURITY INTEREST


In [184]:
#choose the input columns
TEXT_COLUMNS = {
    "COMMUNICATION": ["SUBJECT", "DESCRIPTION"],
    "CLEARANCE": ["PURPOSE"],
    "AOsMEMOs": ["SUBJECT"],
    "LEAVES": ["TYPE OF LEAVE", "REMARKS"],
}

for name, cols in TEXT_COLUMNS.items():
    missing=[c for c in cols if c not in sheets[name].columns]
    print(f"{name:15} using {cols} missing: {missing}")

    

COMMUNICATION   using ['SUBJECT', 'DESCRIPTION'] missing: []
CLEARANCE       using ['PURPOSE'] missing: []
AOsMEMOs        using ['SUBJECT'] missing: []
LEAVES          using ['TYPE OF LEAVE', 'REMARKS'] missing: []


In [185]:
#build document_text without repeated text
def join_unique(values):
    result = ""
    for value in values:
        value =str(value).strip()
        if value == "" or value.lower() == "nan":
            continue
        if value.upper() in result.upper():
            continue
        if result.upper() in value.upper():
            result=value
        else:
            result=(result + " " + value).strip()
    return result

parts = []
for name, cols in TEXT_COLUMNS.items():
    df = sheets[name]
    text = df[cols].fillna("").apply(join_unique, axis=1)
    parts.append(pd.DataFrame({
        "row_id": df["row_id"],
        "document_text": text,
        "label":name,
    }))

datasets=pd.concat(parts, ignore_index=True)

print(datasets.shape)
print(datasets["label"].value_counts())
display(datasets.sample(5, random_state=42))


#this joins each row's chosen columns into one text, skipping repeats, then stacks the 4 sheets into one table called datasets
#check: about 3650 rows; COMMUNICATION ~ 2,285,         

(3654, 3)
label
COMMUNICATION    2285
CLEARANCE         702
LEAVES            653
AOsMEMOs           14
Name: count, dtype: int64


,row_id,document_text,label
1313,COMMUNICATION-1313,"MEMORANDUM ORDER NO. 2025-11-11-001 APPROVED SCHEDULE OF DUTY ASSIGNMENT @ BANGAG, LALLO",COMMUNICATION
1674,COMMUNICATION-1674,REQUEST FOR CHANGE OF PAYEE NAME REQUEST THAT THE ISSUED CHECKS AND FUTURE PAYMENT BE MADE PAYABLE TO NEW TUGUEGARAO BOMBAY BAZAAR CORPORATIOM OR LACHMIII RAICHANDANI OR KUMAR ADTANI,COMMUNICATION
229,COMMUNICATION-229,"AUDIT OBSERVATIONS AND RECOMMENDATIONS(AOM) NO.2024-009(2023) DATED APRIL 1,2024",COMMUNICATION
3509,LEAVES-508,WELLNESS LEAVE,LEAVES
2222,COMMUNICATION-2222,"TRAINING ON DATA ANALYSIS AND VISUALIZATION SOFTWARE ENTITLED""INTERACTIVE DASHBOARD: ""TRANSFORMING DATA INTO DYNAMIC INSIGHTS USING MS POWER BI"" BATCH 2 ON SEPTEMBER 16-18. 2026 AT GVIEW EVENT CENTER",COMMUNICATION


In [186]:
#prove the doubled sentences are gone
comm_text=datasets.loc[datasets["label"]  == "COMMUNICATION", "document_text"]


def has_repeat(text):
    half=len(text) // 2
    return len(text) > 20 and text[:half].strip() == text[half:].strip()

print("texts repeated twice:", comm_text.map(has_repeat).sum())


#checks wheter any COMMUNICATION text is the same sentence written twice
#checks : prints 0 or close to it

texts repeated twice: 0


In [187]:
#clean the text and remove blank rows
def clean_text(text):
    text = str(text).upper()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

datasets["document_text"] = datasets["document_text"].map(clean_text)


blank=datasets["document_text"].eq("")
print("blank_rows:", blank.sum())
display(datasets[blank].head())

datasets=datasets[~blank].reset_index(drop=True)
print("rows after removing blanks:", len(datasets))
print(datasets["label"].value_counts())


#what it does: clean_texts makes every text uppercase and turns runs of spaces, tabs or new lines (\s +) into one space. That way "Sick leave" and "SICK LEAVE" become the same text.
# .map(clean_text) runs the function on every row.
# blank marks the rows whose text is empty (for example, a CLEARANCE row with no PURPOSE). ~blank keeps the rows that are not blank.
# reset_index(drop=True) renumbers the rows after removing some.

blank_rows: 10


,row_id,document_text,label
2292,CLEARANCE-7,,CLEARANCE
2294,CLEARANCE-9,,CLEARANCE
2295,CLEARANCE-10,,CLEARANCE
2296,CLEARANCE-11,,CLEARANCE
2300,CLEARANCE-15,,CLEARANCE


rows after removing blanks: 3644
label
COMMUNICATION    2285
CLEARANCE         693
LEAVES            652
AOsMEMOs           14
Name: count, dtype: int64


In [188]:
#check duplicates and conflicting labels
dupes =  datasets["document_text"].duplicated().sum()
print(dupes)

labels_per_text = datasets.groupby("document_text")["label"].nunique()
conflicts = labels_per_text[labels_per_text > 1]
print("same text with different labels")
display(datasets["document_text"].value_counts().head(10))

print("same text with different lables:", len(conflicts))

#what it does
#duplicated() marks a text that already appeared earlier, and .sum() counts them.
#group groupby("document_text")["label"].nunique() asks, for each distinct text, how many different labels it has. More than 1 is a conflict: the same words labeled two ways.
# value_counts().head(10) shows the 10 most common texts.




1219
same text with different labels


document_text
TRAVEL ABROAD     208
SICK LEAVE        206
COMPENSATORY      127
SPL               108
TERMINAL           95
RETIREMENT         80
MANDATORY          78
WELLNESS LEAVE     47
MONETIZATION       27
VACATION LEAVE     24
Name: count, dtype: int64

same text with different lables: 8


In [189]:
#showing the conflicting text
conflict_rows = datasets[datasets["document_text"].isin(conflicts.index)]

summary = (
    conflict_rows
    .groupby(["document_text", "label"])
    .size()
    .unstack(fill_value=0)
)

print("conflicting texts:", len(summary))
print("rows involved: " , len(conflict_rows))
display(summary.sort_values(summary.columns.tolist(), ascending=False).head(20))

#what it does
#isin(conflicts.index) keeps only the rows whose text is one of the conflicting texts.
# groupby([...]).size() counts the rows per (text, label) pair.
# .unstack(fill_value=0) turns the labels into columns, so you see one row per text, like TERMINAL | CLEARANCE: 90 | LEAVES: 5. fill_value=0 puts 0 where a pair doesn’t exist.

conflicting texts: 8
rows involved:  360


label,AOsMEMOs,CLEARANCE,COMMUNICATION,LEAVES
document_text,,,,
EXECUTIVE ORDER NO. 01 - STREAMLINING AND RATIONALIZING BUREAUCRATIC PROCESSES THROUGH FUNCTIONAL GROUPING OF DEPT. AND OFFICES,1,0,1,0
"MEMORANDUM 34A: REQUEST FOR PERMANENT ALTERNATE REPRESENTATIVE OF THE PMSMEDC, SEND VIA EMAIL ON OR BEFORE MARCH 31, 2025",1,0,1,0
MEMORANDUM ORDER NO. 28 S. 2026 ADOPTION OF LANDBANK WEACCESS ATM PAYROLL AND ACIC FILE UPLOAD FACILITY IN THE PROVINCIAL GOVERNMENT OF CAGAYAN STARTING MAY 2026,1,0,1,0
TERMINAL,0,94,0,1
SICK LEAVE,0,20,0,186
MATERNITY,0,16,0,1
STUDY LEAVE,0,11,0,1
VACATION LEAVE,0,7,0,17


In [190]:
#label review
ORDER_PATTERN = (
    r"\b(?:ADMINISTRATIVE ORDER|EXECUTIVE ORDER|MEMORANDUM ORDER|"
    r"MEMORANDUM NO\.?|MEMORANDUM \d|AO NO\.?|EO NO\.?)"
)
is_order = datasets["document_text"].str.contains(ORDER_PATTERN, regex=True)
candidates = datasets[(datasets["label"] == "COMMUNICATION") & is_order]

print("rows to review:", len(candidates))
display(candidates[["row_id", "document_text"]].head(10))

rows to review: 154


,row_id,document_text
27,COMMUNICATION-27,"MEMORANDUM ORDER NO. 03,2024 IMPLEMENTATION OF THE NEW BIOMETRIC DEVICE"
190,COMMUNICATION-190,"INTERNAL AUDIT MEMORANDUM NO. 01S, 2024 REITERATION OF MEMORANDUM DATED 12/22/2024 ISSUED BY PEO-MOTORPOOL DIVISION RE: GUIDELINES FOR MOTOR VEHICLE REPAIR AND MAINTENANCE"
321,COMMUNICATION-321,"MEMORANDUM ORDER DATED NOVEMBER 13,2024 APPROVED BY GOV. ON THE OPEN PURCHASE REQUESTS"
332,COMMUNICATION-332,AUDIT OBSERVATION MEMORANDUM 2024-040 COLLECTIONS WERE NOT DEPOSITED INTACT TO THE AUTHORIZED GOV. DEPOSITORY BANK
350,COMMUNICATION-350,AUDIT OBSERVATION MEMORANDUM 2024-040
438,COMMUNICATION-438,"EXECUTIVE ORDER NO. 1, SERIES OF 2025,""AN ORDINANCE PRESCRIBING THE GUIDELINES FOR THE REIMBURSEMENT OF COMMUNICATION EXPENSES OF QUALIFIED OFFICIALS/EMPLOYEES OF THE PROVINIAL GOVERNMENT OF CAGAYAN"", OFFICE OF THE GOVERNOR."
467,COMMUNICATION-467,"DOST AO NO. 011, SERIES OF 2020 RE: REVISED GUIDELINES FOR GRANTS-IN-AID PROGRAM FOR DEPARTMENT OF SCIENCE & TECHNOLOGY AND ITS AGENCIES FOR THE LIQUIDATION."
657,COMMUNICATION-657,"FAMILIARIZATION SESSION ON THE PROPER COMPLETION OF THE NEW STANDARDIZED PR TEMPLATE TO BE CONDUCTED ON APRIL 7-8, 2025 @ PEO CONFERENCE HALL CAPITOL COMPOUND, TUGUEGARQAO CITY. ADMINISTRATIVE ORDER NO. 2, SERIES OF 2025, ISSUED BY THE OFFICE OF THE GOVERNOR, TITLED: ""AN ORDER APPROVING AND ADOPTING THE JOINT PROVINCIAL FINANCE COMMITTEE (PFC) - OFFICE OF THE PROVINCIAL ACCOUNTANT - OFFICE OF THE GSO RESOLUTION RECOMMENDING THE USE OF A STANDARDIZED PURCAHSE REQUEST TEMPLATE FOR PROCUREMENT ACTIVITIES TO BE USED BY THE PGC"
686,COMMUNICATION-686,"DEPARTMENT MEMORANDUM NO. 2024-0462: ADVISORY ON THE IMPLEMETATION OF THE MEDICAL ASSISTANCE TO INDIGENT AND FINANCIALLY INCAPACITATED PATIENTS (MAIFIP) PROGRAM IN CONNECTION WITH THE MAY 12, 2025 NATIONAL ANAD LOCAL ELECTIONS, AND BARMM PARLIAMENTARY ELECTIONS"
772,COMMUNICATION-772,"INTERNAL AUDIT MEMORANDUM ORDER NO.08, S.2025: AGREEMENT ON STANDARDIZATION OF DOCUMENTARY REQUIREMENTS FOR FUND CONTROL PURPOSES"


In [191]:
#Create the review file and carry over your 11 old decisions
OLD_REVIEW_FILE = Path("../data/interim/pao_label_review_old.csv")

if REVIEW_FILE.exists():
    review = pd.read_csv(REVIEW_FILE, keep_default_na=False).set_index("row_id")
    print("loaded saved review")
else:
    review = candidates[["row_id", "document_text", "label"]].rename(
        columns={"label": "original_label"}).set_index("row_id")
    review["reviewed_label"] = ""
    review["review_status"] = "pending"

    # carry over decisions from the old notebook (old id 27 -> COMMUNICATION-27)
    old = pd.read_csv(OLD_REVIEW_FILE, keep_default_na=False)
    old = old[old["review_status"] != "pending"]
    old["row_id"] = "COMMUNICATION-" + old["row_id"].astype(str)
    old = old.set_index("row_id")

    found = old.index.intersection(review.index)
    review.loc[found, ["reviewed_label", "review_status"]] = \
        old.loc[found, ["reviewed_label", "review_status"]]
    print(f"carried over {len(found)} of {len(old)} old decisions")

    review.to_csv(REVIEW_FILE)
    print("created new review file")

print(review["review_status"].value_counts())

# What it does:
# The first time, it builds the review table from candidates, copies your old decisions across, and saves it to the CSV.
# Every time after that, it just loads the CSV, so your progress is never lost.
# The ID conversion: your old file used number IDs like 27. In the old notebook those were COMMUNICATION’s row positions, which are the same positions your new row_id uses, so 27 becomes COMMUNICATION-27.
# index.intersection(...) keeps only the IDs that exist in both tables, so no row is matched by mistake.

loaded saved review
review_status
pending                128
verified                22
needs_domain_review      4
Name: count, dtype: int64


In [192]:
#Confirm the carried-over decisions point at the right rows
done = review[review["review_status"] != "pending"]
display(done[["document_text", "reviewed_label", "review_status"]])

,document_text,reviewed_label,review_status
row_id,,,
COMMUNICATION-27,"MEMORANDUM ORDER NO. 03,2024 IMPLEMENTATION OF THE NEW BIOMETRIC DEVICE",AOsMEMOs,verified
COMMUNICATION-190,"INTERNAL AUDIT MEMORANDUM NO. 01S, 2024 REITERATION OF MEMORANDUM DATED 12/22/2024 ISSUED BY PEO-MOTORPOOL DIVISION RE: GUIDELINES FOR MOTOR VEHICLE REPAIR AND MAINTENANCE",,needs_domain_review
COMMUNICATION-321,"MEMORANDUM ORDER DATED NOVEMBER 13,2024 APPROVED BY GOV. ON THE OPEN PURCHASE REQUESTS",AOsMEMOs,verified
COMMUNICATION-332,AUDIT OBSERVATION MEMORANDUM 2024-040 COLLECTIONS WERE NOT DEPOSITED INTACT TO THE AUTHORIZED GOV. DEPOSITORY BANK,COMMUNICATION,verified
COMMUNICATION-350,AUDIT OBSERVATION MEMORANDUM 2024-040,COMMUNICATION,verified
COMMUNICATION-438,"EXECUTIVE ORDER NO. 1, SERIES OF 2025,""AN ORDINANCE PRESCRIBING THE GUIDELINES FOR THE REIMBURSEMENT OF COMMUNICATION EXPENSES OF QUALIFIED OFFICIALS/EMPLOYEES OF THE PROVINIAL GOVERNMENT OF CAGAYAN"", OFFICE OF THE GOVERNOR.",AOsMEMOs,verified
COMMUNICATION-467,"DOST AO NO. 011, SERIES OF 2020 RE: REVISED GUIDELINES FOR GRANTS-IN-AID PROGRAM FOR DEPARTMENT OF SCIENCE & TECHNOLOGY AND ITS AGENCIES FOR THE LIQUIDATION.",,needs_domain_review
COMMUNICATION-657,"FAMILIARIZATION SESSION ON THE PROPER COMPLETION OF THE NEW STANDARDIZED PR TEMPLATE TO BE CONDUCTED ON APRIL 7-8, 2025 @ PEO CONFERENCE HALL CAPITOL COMPOUND, TUGUEGARQAO CITY. ADMINISTRATIVE ORDER NO. 2, SERIES OF 2025, ISSUED BY THE OFFICE OF THE GOVERNOR, TITLED: ""AN ORDER APPROVING AND ADOPTING THE JOINT PROVINCIAL FINANCE COMMITTEE (PFC) - OFFICE OF THE PROVINCIAL ACCOUNTANT - OFFICE OF THE GSO RESOLUTION RECOMMENDING THE USE OF A STANDARDIZED PURCAHSE REQUEST TEMPLATE FOR PROCUREMENT ACTIVITIES TO BE USED BY THE PGC",COMMUNICATION,verified
COMMUNICATION-686,"DEPARTMENT MEMORANDUM NO. 2024-0462: ADVISORY ON THE IMPLEMETATION OF THE MEDICAL ASSISTANCE TO INDIGENT AND FINANCIALLY INCAPACITATED PATIENTS (MAIFIP) PROGRAM IN CONNECTION WITH THE MAY 12, 2025 NATIONAL ANAD LOCAL ELECTIONS, AND BARMM PARLIAMENTARY ELECTIONS",COMMUNICATION,verified


In [193]:
# Helper to mark rows and save
def mark(row_ids, label=None, status="verified"):
    if status == "verified" and label is None:
        raise ValueError("verified rows need a label")
    review.loc[row_ids, "review_status"] = status
    if label:
        review.loc[row_ids, "reviewed_label"] = label
    review.to_csv(REVIEW_FILE)
    print(review["review_status"].value_counts().to_dict())

In [194]:
pending = review[review["review_status"] == "pending"]
print("pending left:", len(pending))
display(pending[["document_text"]].head(15))


pending left: 128


,document_text
row_id,
COMMUNICATION-1107,"MEMORANDUM ORDER NO. 020 AS PER THE GOVERNORS ORDER, EACH DEPARTMENT SHALL CONFER AND/OR COORDINATE WITH THE PROVINCIAL ADMINISTRATOR EACH DEPARTMENTS PPA IMPLEMENTATION AND ANY INSTRUCTION FROM THE GOVERNOR. OFFFICES/DIVISIONS UNDER GO MAY COPY FURNISH/NOTIFY THE PA'S OFFICE OF THEIR ACTIVITIES OR SEEK ADVISE IF NEEDED"
COMMUNICATION-1120,"MEMORANDUM ORDER NO. 37, S.2025 SUSPENSION OF OFFICIAL TRAVEL OUTSIDE THE PROVINCE DUE TO TROPICAL CYCLONE MIRASOL"
COMMUNICATION-1124,"MEMORANDUM ORDER NO. 023, S.2025 ENDORSEMENET OF/CONSULTATION REQUIREMENT FOR ICT EQUIPMENT PROCUREMENT"
COMMUNICATION-1125,"MEMORANDUM NO. 280, S.2025 ENDORSEMENT OF REPORTS FOR THE UTILIZED LOCAL GOVERNMENT SUPPORT FUND - FA TO LGU (LGSF - FALGU) FOR REVIEW AND APPROVAL OF BOSS. JEANNA GARMA"
COMMUNICATION-1135,"ADMINISTRATIVE ORDER NO. 02, S.2025 AN ORDER CREATING TWO (2) AD-HOC COMMITTEES TO CONDUCT A COMPREHENSIVE, ACTUAL, AND PHYSICAL INVENTORY OF ALL PROPERTIES OWNED BY THE PROVINCE OF CAGAYAN AD-HOC COMMITTEE 1: 1. RODALYN ROBLES - GSO 2. KERNEL BALIGOD - CPLRC 3. ARNOLD LANGCAY - CPLRC AD-HOC COMMITTE 2: 1. SHERWIN LIM - GSO 2. RALPHLIN PASCUA - ACCOUNTING 3. URSULA MADRID - GSO"
COMMUNICATION-1138,"MEMORANDUM NO. 41, S.2025 TO: MR. OLIVER B. PENEYRA, POLITICAL CONSULTANT - GO MEMORANDUM NO. 42, S.2025 TO: MR. ATANACIO T. MACALAN, EXECUTIVE ASSISTANT III - GO ADDITIONAL DUTIES AND RESPONSIBILITIES: TO ENSURE THE SAFETY, SECURITY , AND ORDERLINESS OF THE SUB-CAPITOL COMPLEX IN BANGAG, LALLO, CAGAYAN ADDITIONAL DUTIES AND RESPONSIBILITIES: TO ENSURE THE SAFETY, SECURITY , AND ORDERLINESS OF THEPROVINCIAL CAPITOL MAIN COMPLEX IN TUGUEGARAO CITY"
COMMUNICATION-1139,"MEMORANDUM NO. 86, S.2025: SUB-CAPITOL MANPOWER SUPPLEMENT DIRECTIVE TO SUBMIT RESPECTIVE MANPOWER SUPPLEMENT AND SERVICE PROFILE TO ENSURE EFFECTIVE AND EFFICIENT SERVICE DELIVERY IN THE PROVINCIAL SUB-CAPITOL IN LAL-LO, BANGAG, CAGAYAN"
COMMUNICATION-1145,"MEMORANDUM NO. 43, S.2025 PROVINCIAL DEVELOPMENT COUNCIL MEETING"
COMMUNICATION-1155,"MEMORANDUM ORDER NO. 44, S.2025 PROVINCIAL DEVELOPMENT COUNCIL MEETING - EXECUTIVE COMMITTEE ON SEPTEMBER 26, 2025, 8:00AM 1F, GO, MAIN CONFERENCE HALL"


In [195]:
mark(["COMMUNICATION-438", "COMMUNICATION-840", "COMMUNICATION-851",
      "COMMUNICATION-882", "COMMUNICATION-898", "COMMUNICATION-996",
      "COMMUNICATION-1005", "COMMUNICATION-1057"], "AOsMEMOs")

mark(["COMMUNICATION-332", "COMMUNICATION-350",
      "COMMUNICATION-657", "COMMUNICATION-686"], "COMMUNICATION")

mark(["COMMUNICATION-190", "COMMUNICATION-1008",
      "COMMUNICATION-1009"], status="needs_domain_review")

{'pending': 128, 'verified': 22, 'needs_domain_review': 4}
{'pending': 128, 'verified': 22, 'needs_domain_review': 4}
{'pending': 128, 'verified': 22, 'needs_domain_review': 4}


In [196]:
ask_pao = review[review["review_status"] == "needs_domain_review"]

ask_file = Path("../data/interim/pao_ask_domain_expert.csv")
ask_pao[["document_text", "original_label"]].to_csv(ask_file)

print(len(ask_pao), "rows saved to", ask_file)
display(ask_pao[["document_text"]])


4 rows saved to ..\data\interim\pao_ask_domain_expert.csv


,document_text
row_id,
COMMUNICATION-190,"INTERNAL AUDIT MEMORANDUM NO. 01S, 2024 REITERATION OF MEMORANDUM DATED 12/22/2024 ISSUED BY PEO-MOTORPOOL DIVISION RE: GUIDELINES FOR MOTOR VEHICLE REPAIR AND MAINTENANCE"
COMMUNICATION-467,"DOST AO NO. 011, SERIES OF 2020 RE: REVISED GUIDELINES FOR GRANTS-IN-AID PROGRAM FOR DEPARTMENT OF SCIENCE & TECHNOLOGY AND ITS AGENCIES FOR THE LIQUIDATION."
COMMUNICATION-1008,"MEMORANDUM NO. 013 S. 2025 PRIORITIZATION OF SGLG-RELATED PROGRAMS IN ANNUAL OPERATIONS PLAN, (AOP) AND ANNUAL INVESTMENT PLAN (AIP) 2026"
COMMUNICATION-1009,MEMORANDUM NO. 012 S. 2025 CREATION OF SGLG TECHNICAL WORKING GROUP


In [197]:
assert (review["review_status"] == "pending").sum() == 0, "finish the review first"

verified = review[review["review_status"] == "verified"]

datasets["original_label"] = datasets["label"]
datasets = datasets.set_index("row_id")
datasets.loc[verified.index, "label"] = verified["reviewed_label"]
datasets = datasets.reset_index()

print("labels changed:", (datasets["label"] != datasets["original_label"]).sum())
display(pd.crosstab(datasets["original_label"], datasets["label"]))
print(datasets["label"].value_counts())

AssertionError: finish the review first

In [ ]:
labels_per_text = datasets.groupby("document_text")["label"].nunique()
conflicts_after = labels_per_text[labels_per_text > 1]

print("conflicts before review: 8")
print("conflicts after review: ", len(conflicts_after))
display(conflicts_after.index.to_series().head(10))

conflicts before review: 8
conflicts after review:  8


document_text
EXECUTIVE ORDER NO. 01 - STREAMLINING AND RATIONALIZING BUREAUCRATIC PROCESSES THROUGH FUNCTIONAL GROUPING OF DEPT. AND OFFICES                                                                        EXECUTIVE ORDER NO. 01 - STREAMLINING AND RATIONALIZING BUREAUCRATIC PROCESSES THROUGH FUNCTIONAL GROUPING OF DEPT. AND OFFICES
MATERNITY                                                                                                                                                                                                                                                                                                                    MATERNITY
MEMORANDUM 34A: REQUEST FOR PERMANENT ALTERNATE REPRESENTATIVE OF THE PMSMEDC, SEND VIA EMAIL ON OR BEFORE MARCH 31, 2025                                                                                    MEMORANDUM 34A: REQUEST FOR PERMANENT ALTERNATE REPRESENTATIVE OF THE PMSMEDC, SEND VIA EMAIL ON OR BEFORE MARCH 31, 2025
MEMOR

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# stop early with a clear message if Cell 15 has not applied the reviewed labels yet
assert "original_label" in datasets.columns, "run Cell 15 first (finish the review)"

final = datasets[["row_id", "document_text", "label", "original_label"]]
final.to_parquet(OUTPUT_FILE, index=False)

check = pd.read_parquet(OUTPUT_FILE)
print(check.shape)
print(check["label"].value_counts())

KeyError: "['original_label'] not in index"